# CPI Backtest Demo

End-to-end backtest comparing two predictors on the CPI All-items Canada
12-month ahead forecasting task.

**Task:** Predict Canada CPI All-items (2002=100) exactly 12 months ahead.  
**Origins:** January and July, 2002–2026 (twice per year).  
**Predictors:**
- `LastValuePredictor` — naive last-value baseline from `methods.naive`; the performance floor every predictor must beat
- `DartsAutoARIMAPredictor` — a `Predictor` subclass wrapping Darts `AutoARIMA`, defined inline below

**Score:** CRPS (lower is better).

`DartsAutoARIMAPredictor` is defined in full in section 3 so you can see
what a Darts-based implementation looks like. To write your own predictor,
read `implementations/methods/naive.py` for an annotated structural reference,
then subclass `Predictor` directly.

## Prerequisites

Populate the local data cache before running this notebook:

```bash
uv run python scripts/fetch_cpi.py
```

In [ ]:
import sys
from pathlib import Path

import yaml


# Ensure the workspace root is on sys.path when run from this directory.
# Notebook lives at implementations/experiments/economic_forecasting/ — 3 levels deep.
repo_root = Path().resolve().parents[2]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## 1. Register CPI All-items

In [ ]:
from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters import StatCanAdapter


CACHE_DIR = repo_root / "data" / "statcan"

svc = DataService()

adapter = StatCanAdapter(
    table_id="18-10-0004-11",
    member_filter={"GEO": "Canada", "Products and product groups": "All-items"},
    cache_dir=CACHE_DIR,
)
meta = SeriesMetadata(
    series_id="cpi_all_items_canada",
    description="CPI All-items, Canada (2002=100)",
    source="StatCan table 18-10-0004-11",
    units="Index 2002=100",
    frequency="MS",
)
svc.register("cpi_all_items_canada", adapter, meta)
svc.summary()

## 2. Load the reference spec

In [ ]:
from aieng.forecasting.evaluation import BacktestSpec


spec_path = repo_root / "reference_specs" / "cpi_allitems_12m.yaml"
with spec_path.open() as f:
    spec = BacktestSpec.model_validate(yaml.safe_load(f))

origins = spec.origins()
print(f"Task:    {spec.task.task_id}")
print(f"Horizon: {spec.task.horizon} months")
print(f"Origins: {len(origins)} ({origins[0].date()} → {origins[-1].date()})")
print(f"Warmup:  {spec.warmup} observations")

## 3. Define the predictors

We compare two predictors:

- **`LastValuePredictor`** — imported directly from `methods.naive`. No fitting required; predicts the last observed value at all quantiles. This is the performance floor.
- **`DartsAutoARIMAPredictor`** — defined inline below. Wraps Darts `AutoARIMA` with Monte Carlo sampling for probabilistic forecasts.

The key steps for any `Predictor` implementation:
1. Implement `predictor_id` — a stable string identifier for this predictor.
2. Inside `predict()`, fetch the target series from `context` (cut off at `context.as_of`), fit your model, and extract a probabilistic forecast.
3. Return a `Prediction` containing a `ContinuousForecast` (point estimate + quantiles).

In [ ]:
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from aieng.forecasting.data.context import ForecastContext
from aieng.forecasting.evaluation.prediction import STANDARD_QUANTILES, ContinuousForecast, Prediction
from aieng.forecasting.evaluation.predictor import Predictor
from aieng.forecasting.evaluation.task import ForecastingTask
from methods.naive import LastValuePredictor


class DartsAutoARIMAPredictor(Predictor):
    """Probabilistic predictor wrapping Darts AutoARIMA.

    Fits AutoARIMA on the target series history available at the forecast
    origin, then generates a probabilistic forecast via Monte Carlo sampling.
    """

    def __init__(self, num_samples: int = 500, predictor_id: str = "darts_auto_arima") -> None:
        self._num_samples = num_samples
        self._predictor_id = predictor_id

    @property
    def predictor_id(self) -> str:
        return self._predictor_id

    def predict(self, task: ForecastingTask, context: ForecastContext) -> Prediction:
        # Lazy import: darts is heavy; only load when predicting.
        from darts import TimeSeries  # noqa: PLC0415
        from darts.models import AutoARIMA  # noqa: PLC0415

        series_df = context.get_series(task.target_series_id)

        # Convert to a regular Darts TimeSeries, forward-filling any gaps.
        ts = TimeSeries.from_dataframe(
            series_df,
            time_col="timestamp",
            value_cols="value",
            fill_missing_dates=True,
            freq=task.frequency,
        )

        model = AutoARIMA()
        model.fit(ts)

        # Generate probabilistic forecast via Monte Carlo sampling.
        forecast_ts = model.predict(n=task.horizon, num_samples=self._num_samples)

        # forecast_ts.all_values() shape: (horizon, n_components, n_samples).
        # Take the final step — that is the horizon target.
        samples: np.ndarray = forecast_ts.all_values()[-1, 0, :]

        point_forecast = float(np.median(samples))
        quantiles = {q: float(np.quantile(samples, q)) for q in STANDARD_QUANTILES}

        forecast_date: datetime = (
            pd.Timestamp(context.as_of) + pd.tseries.frequencies.to_offset(task.frequency) * task.horizon
        ).to_pydatetime()

        payload = ContinuousForecast(point_forecast=point_forecast, quantiles=quantiles)

        return Prediction(
            predictor_id=self.predictor_id,
            task_id=task.task_id,
            issued_at=datetime.now(tz=timezone.utc).replace(tzinfo=None),
            as_of=context.as_of,
            forecast_date=forecast_date,
            payload=payload,
        )


naive_predictor = LastValuePredictor()
arima_predictor = DartsAutoARIMAPredictor(num_samples=500)
print(f"Predictors: {naive_predictor.predictor_id}, {arima_predictor.predictor_id}")

## 4. Run both backtests

The naive backtest is instant. AutoARIMA fits once per forecast origin — expect a few minutes.

In [ ]:
from aieng.forecasting.evaluation import backtest


naive_results = backtest(predictor=naive_predictor, spec=spec, data_service=svc)
arima_results = backtest(predictor=arima_predictor, spec=spec, data_service=svc)

print(f"{'Predictor':<30} {'Origins':>8} {'Skipped':>8} {'Mean CRPS':>10}")
print("-" * 60)
for r in [naive_results, arima_results]:
    print(f"{r.predictor_id:<30} {len(r.predictions):>8} {r.skipped_origins:>8} {r.mean_crps:>10.4f}")

## 5. Per-origin CRPS comparison

In [ ]:
def result_to_df(result, label):
    return pd.DataFrame(
        {
            "origin": [p.as_of.date() for p in result.predictions],
            "forecast_date": [p.forecast_date.date() for p in result.predictions],
            f"point_{label}": [p.payload.point_forecast for p in result.predictions],
            f"crps_{label}": result.scores,
        }
    ).set_index("forecast_date")


naive_df = result_to_df(naive_results, "naive")
arima_df = result_to_df(arima_results, "arima")

comparison = naive_df.join(arima_df[["point_arima", "crps_arima"]], how="inner")
comparison["crps_reduction"] = comparison["crps_naive"] - comparison["crps_arima"]

print(f"Mean CRPS  naive: {comparison['crps_naive'].mean():.4f}")
print(f"Mean CRPS  arima: {comparison['crps_arima'].mean():.4f}")
print(
    f"Mean reduction:   {comparison['crps_reduction'].mean():.4f}  ({comparison['crps_reduction'].mean() / comparison['crps_naive'].mean() * 100:.1f}%)"
)
comparison.reset_index()

## 6. Predictions vs. actuals — AutoARIMA vs. naive

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt


COLOR_OBS = "#1a1a2e"
COLOR_ARIMA = "#e94560"
COLOR_NAIVE = "#f5a623"

# Retrieve the full observed series for plotting.
full_series = svc.get_series(
    "cpi_all_items_canada",
    as_of=datetime.now(tz=timezone.utc).replace(tzinfo=None),
)

arima_dates = [p.forecast_date for p in arima_results.predictions]
arima_points = [p.payload.point_forecast for p in arima_results.predictions]
arima_q10 = [p.payload.quantiles.get(0.10, p.payload.point_forecast) for p in arima_results.predictions]
arima_q90 = [p.payload.quantiles.get(0.90, p.payload.point_forecast) for p in arima_results.predictions]

naive_dates = [p.forecast_date for p in naive_results.predictions]
naive_points = [p.payload.point_forecast for p in naive_results.predictions]

fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [3, 1]})

# --- Top panel: observed series + both predictors ---
ax = axes[0]
ax.plot(full_series["timestamp"], full_series["value"], color=COLOR_OBS, linewidth=1.5, label="Observed CPI")
ax.fill_between(arima_dates, arima_q10, arima_q90, alpha=0.20, color=COLOR_ARIMA, label="AutoARIMA 80% CI")
ax.scatter(
    arima_dates,
    arima_points,
    color=COLOR_ARIMA,
    s=22,
    zorder=5,
    label=f"AutoARIMA  (mean CRPS {arima_results.mean_crps:.3f})",
)
ax.scatter(
    naive_dates,
    naive_points,
    color=COLOR_NAIVE,
    s=18,
    marker="^",
    zorder=4,
    label=f"Naive       (mean CRPS {naive_results.mean_crps:.3f})",
)
ax.set_title(
    "CPI All-items Canada — 12-month ahead backtest: AutoARIMA vs. naive baseline", fontsize=13, fontweight="bold"
)
ax.set_ylabel("CPI Index (2002=100)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)

# --- Bottom panel: per-origin CRPS for both predictors ---
ax2 = axes[1]
ax2.plot(
    naive_dates,
    naive_results.scores,
    color=COLOR_NAIVE,
    linewidth=1.2,
    marker="^",
    markersize=4,
    label=f"Naive  (mean {naive_results.mean_crps:.3f})",
)
ax2.plot(
    arima_dates,
    arima_results.scores,
    color=COLOR_ARIMA,
    linewidth=1.2,
    marker="o",
    markersize=4,
    label=f"AutoARIMA  (mean {arima_results.mean_crps:.3f})",
)
ax2.axhline(naive_results.mean_crps, color=COLOR_NAIVE, linestyle="--", linewidth=0.8, alpha=0.6)
ax2.axhline(arima_results.mean_crps, color=COLOR_ARIMA, linestyle="--", linewidth=0.8, alpha=0.6)
ax2.set_ylabel("CRPS")
ax2.set_xlabel("Forecast date (resolution)")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.xaxis.set_major_locator(mdates.YearLocator(2))
ax2.legend()
ax2.grid(axis="y", linestyle="--", alpha=0.4)

fig.tight_layout()
plt.show()

## 7. Serialize the AutoARIMA result to YAML

`BacktestResult` is a Pydantic model — it can be serialized to YAML and
persisted alongside the predictor implementation, passed to a downstream
agent as structured context, or used as a submission artefact. The same
applies to `naive_results`.

In [ ]:
result_dict = arima_results.model_dump(mode="json")
result_yaml = yaml.dump(result_dict, default_flow_style=False, allow_unicode=True)

# Preview the first ~40 lines.
print("\n".join(result_yaml.splitlines()[:40]))
print("...")